# Assistant permis de construire — San Francisco : notebook de démonstration

Chaque brique de l'agent, **sans passer par le LLM** : l'API DataSF, la base des projets du cabinet (fictif), le RAG sur les pages officielles de la ville, le serveur MCP, les skills, les garde-fous et la validation humaine. Puis les résultats de l'évaluation.

> ⚠️ Information générale, pas un avis juridique. Vérifier auprès du Department of Building Inspection.

In [1]:
import os, shutil, sys, tempfile, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT)); os.chdir(ROOT)
os.environ.setdefault('RAG_EMBEDDINGS', 'local')  # 'gemini' si une clé est configurée
from app import config
tmp = Path(tempfile.mkdtemp())  # copie de la base : le notebook ne modifie rien
shutil.copy(config.ROOT/'data'/'projects.db', tmp/'projects.db')
config.PROJECTS_DB_PATH = tmp/'projects.db'; config.CHROMA_DIR = tmp/'chroma'
config.DATASF_CACHE_PATH = tmp/'datasf.sqlite'
import pandas as pd, sqlite3
con = sqlite3.connect(config.PROJECTS_DB_PATH)

## 1. API DataSF « Building Permits » (permis existants à une adresse)

In [2]:
from app import tools
r = tools.search_permits('781 20th Ave')
print(r['address'], '|', r['neighborhood'], '|', r['permit_count'], 'permis | source :', r['source'])
pd.DataFrame(r['permits'])[['permit_number','permit_type_definition','status','status_date','estimated_cost']]

781 20th Av | Outer Richmond | 8 permis | source : DataSF — Building Permits (i98e-djp9), https://data.sf.gov/d/i98e-djp9


,permit_number,permit_type_definition,status,status_date,estimated_cost
0,202604270175,otc alterations permit,issued,2026-07-27T11:26:44.000,1.0
1,202508263948,otc alterations permit,issued,2026-02-24T12:43:25.000,150000.0
2,202507312004,additions alterations or repairs,withdrawn,2025-09-18T00:00:00.000,80000.0
3,202506168765,otc alterations permit,withdrawn,2025-09-16T17:00:26.000,200000.0
4,202502100025,otc alterations permit,reinstated,2025-09-09T15:50:43.000,150000.0
5,202508062425,otc alterations permit,issued,2025-08-21T15:00:59.000,150000.0
6,201006225012,otc alterations permit,complete,2011-01-11T00:00:00.000,20000.0
7,9011117,otc alterations permit,complete,1990-10-23T00:00:00.000,7335.0


In [3]:
print(tools.search_permits('781 20th Ave')['from_cache'])  # 2e appel : cache local
print(tools.search_permits('99999 Market St'))           # adresse introuvable
config.DATASF_URL, url = 'http://127.0.0.1:9/x.json', config.DATASF_URL; config.DATASF_CACHE = False
print(tools.search_permits('2460 Lyon St'))              # API en panne
config.DATASF_URL = url; config.DATASF_CACHE = True

True


{'error': 'ADDRESS_NOT_FOUND', 'message': "Aucun permis trouvé dans DataSF pour « 99999 Market St ». L'adresse est peut-être inexistante ou mal orthographiée (format attendu : numéro + rue, ex. 781 20th Ave)."}
{'error': 'DATASF_UNAVAILABLE', 'message': "L'API DataSF est indisponible (ConnectError)."}


## 2. La base des projets du cabinet (fictif, adresses réelles)

In [4]:
pd.read_sql('SELECT id, client, address, work_type, estimated_cost, status FROM projects', con)

,id,client,address,work_type,estimated_cost,status
0,1,Client fictif A. Moreau,781 20th Ave,kitchen_remodel,65000.0,ready_to_file
1,2,Client fictif B. Chen,2460 Lyon St,kitchen_remodel,120000.0,design
2,3,Client fictif C. Duarte,153 Sickles Ave,kitchen_remodel,38000.0,ready_to_file
3,4,Client fictif D. Okafor,2126 18th St,kitchen_remodel,85000.0,design
4,5,Client fictif E. Larsen,1900 15th Ave,bathroom_remodel,30000.0,ready_to_file
5,6,Client fictif F. Romero,245 Silver Ave,bathroom_remodel,42000.0,design
6,7,Client fictif G. Patel,2542 Bryant St,bathroom_remodel,55000.0,filed
7,8,Client fictif H. Kowalski,40 Sycamore St,adu,280000.0,design
8,9,Client fictif I. Nakamura,30 Jersey St,adu,350000.0,ready_to_file
9,10,Client fictif J. Haddad,1924 Kirkham St,adu,210000.0,design


In [5]:
pd.read_sql('SELECT id, project_id, status, created_at FROM submissions', con)

,id,project_id,status,created_at
0,1,5,pending_approval,2026-11-18T10:00:00+00:00
1,2,12,approved,2026-11-17T09:00:00+00:00
2,3,7,sent,2026-11-10T09:00:00+00:00


## 3. RAG sur les pages officielles de la ville (Chroma)

In [6]:
from app.rag import load_chunks, search_city_docs
chunks = load_chunks(); print(len(chunks), 'morceaux indexés')
res = search_city_docs('documents required for an ADU building permit')
res if 'error' in res else [(p['title'], p['url'], p['score']) for p in res['passages']]

0 morceaux indexés


{'error': 'CORPUS_EMPTY', 'message': "Aucun document officiel n'est indexé."}

## 4. Le serveur MCP

In [7]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
params = StdioServerParameters(command=sys.executable, args=['-m','mcp_server.server'], cwd=str(ROOT),
    env={**os.environ, 'PYTHONPATH': str(ROOT), 'PROJECTS_DB_PATH': str(config.PROJECTS_DB_PATH)})
async def list_tools():
    async with stdio_client(params) as (r, w), ClientSession(r, w) as s:
        await s.initialize(); return [(t.name, t.description.splitlines()[0]) for t in (await s.list_tools()).tools]
await list_tools()

[('search_permits',
  'Permis de construire enregistrés par la ville de San Francisco à une adresse'),
 ('list_projects',
  'Liste des projets du cabinet (id, client, adresse, type de travaux, statut). Lecture seule.'),
 ('get_project',
  "Détail d'un projet du cabinet et de ses dossiers déjà créés. Lecture seule."),
 ('create_submission_draft',
  "Enregistre le brouillon de dossier d'un projet, au statut « pending_approval »."),
 ('send_submission',
  "ACTION RISQUÉE. Envoie un dossier (envoi simulé). Refusé si le dossier n'a pas été")]

## 5. Skills et constructeur de contexte

In [8]:
from types import SimpleNamespace
from app.skills import SKILL_REGISTRY, build_instruction
for s in SKILL_REGISTRY.values(): print(f'{s.name:22} -> {s.allowed_tools}')
ctx = SimpleNamespace(state={'active_skill': 'submission_prep', 'case': {'project_id': 9,
    'address': '30 Jersey St', 'work_type': 'adu'}})
print(build_instruction(ctx)[-1200:])

permit_search          -> ('search_permits',)
document_requirements  -> ('search_city_docs',)
submission_prep        -> ('list_projects', 'get_project', 'search_city_docs', 'search_permits', 'create_submission_draft', 'send_submission')
ssion_prep : Préparer le brouillon de dossier d'un projet du cabinet (base SQL + documents officiels).

Skill active : submission_prep
Instructions :
1. list_projects si l'identifiant du projet n'est pas connu, sinon get_project.
2. search_city_docs pour les documents requis selon le work_type du projet. search_permits sur l'adresse est optionnel (permis existants).
3. Rédige le brouillon en markdown : projet (client, adresse, travaux, coût estimé), procédure probable, liste des pièces à fournir avec pour chacune sa source [titre](url), permis existants éventuels, l'avertissement juridique. Puis appelle create_submission_draft(project_id, content).
4. Le brouillon est EN ATTENTE DE VALIDATION HUMAINE : dis-le. Ne l'envoie pas.
5. send_submission seuleme

## 6. Garde-fous

In [9]:
from app.guardrails import check_tool_call, enforce_output
state = {'active_skill': 'submission_prep'}
draft = {'project_id': 9, 'content': 'Brouillon sans aucune source. ' * 10}
print(check_tool_call('create_submission_draft', draft, state, 'i1'))  # refus : aucune source
print(check_tool_call('send_submission', {'submission_id': 1}, state, 'i1'))  # refus : non approuvé
print(check_tool_call('search_permits', {'address': '1 Main'}, {'active_skill': 'document_requirements'}, 'i1'))
print(enforce_output('Réponse sans avertissement.', None))

guardrail GUARDRAIL_NO_SOURCE on create_submission_draft: Le brouillon doit citer au moins une source officielle (URL) retrouvée par search_city_docs. Sources disponibles : aucune.


guardrail GUARDRAIL_NOT_APPROVED on send_submission: Envoi refusé : le dossier 1 est au statut « pending_approval ». Seul un dossier approuvé par un humain dans l'interface peut être envoyé.


guardrail GUARDRAIL_SKILL on search_permits: L'outil search_permits n'est pas autorisé dans la skill active (document_requirements). Active d'abord la skill 'permit_search'.


{'error': 'GUARDRAIL_NO_SOURCE', 'message': 'Le brouillon doit citer au moins une source officielle (URL) retrouvée par search_city_docs. Sources disponibles : aucune.', 'blocked_by_guardrail': True}
{'error': 'GUARDRAIL_NOT_APPROVED', 'message': "Envoi refusé : le dossier 1 est au statut « pending_approval ». Seul un dossier approuvé par un humain dans l'interface peut être envoyé.", 'blocked_by_guardrail': True}
{'error': 'GUARDRAIL_SKILL', 'message': "L'outil search_permits n'est pas autorisé dans la skill active (document_requirements). Active d'abord la skill 'permit_search'.", 'blocked_by_guardrail': True}
Réponse sans avertissement.

⚠️ Information générale, pas un avis juridique. Vérifier auprès du Department of Building Inspection.


## 7. Validation humaine

In [10]:
from app import db
print(tools.send_submission(1))                 # en attente -> refusé
print(db.decide_submission(1, approve=True, reviewer='architecte référent'))
print(tools.send_submission(1))                 # approuvé -> envoyé (simulé)

{'error': 'NOT_APPROVED', 'message': "Envoi refusé : le dossier 1 est au statut « pending_approval ». Il doit d'abord être approuvé par un humain dans l'interface."}
{'submission_id': 1, 'status': 'approved'}
{'submission_id': 1, 'status': 'sent', 'sent_at': '2026-11-20T16:38:54+00:00', 'message': 'Dossier envoyé (envoi simulé : aucun dépôt réel auprès de la ville).'}


## 8. Évaluation : assistant vs assistant à simple prompt
Produite par `bash scripts/run_eval.sh`.

In [11]:
from IPython.display import Markdown
p = ROOT/'docs'/'eval_results.md'
Markdown(p.read_text() if p.exists() else '_Lancer `bash scripts/run_eval.sh` pour produire les résultats._')

_Lancer `bash scripts/run_eval.sh` pour produire les résultats._